# APIM을 통한 Azure OpenAI 모델 호출

기존 Azure OpenAI 직접 호출 코드를 **APIM 엔드포인트**로 전환하는 방법을 안내합니다.

## 변경 사항 요약

| 항목 | 기존 (직접 호출) | 변경 (APIM 경유) |
|------|----------------|-----------------|
| Endpoint | `https://{resource}.openai.azure.com` | `https://{apim-name}.azure-api.net` |
| API Key | Foundry Project API Key | **APIM Subscription Key** |
| 코드 변경 | - | `azure_endpoint`과 `api_key`만 변경 |

## 사전 준비

1. `.env` 파일에 아래 값을 설정하세요:
   - `APIM_ENDPOINT`: APIM 게이트웨이 URL
   - `APIM_SUBSCRIPTION_KEY`: 팀에서 공유받은 APIM Subscription Key
   - `DEPLOYMENT_NAME`: 사용할 모델 배포 이름 (예: gpt-4o)
   - `API_VERSION`: API 버전 (예: 2024-12-01-preview)

In [ ]:
# 환경 변수 로드
import os
from dotenv import load_dotenv

load_dotenv()

apim_endpoint = os.getenv("APIM_ENDPOINT")
apim_key = os.getenv("APIM_SUBSCRIPTION_KEY")
deployment_name = os.getenv("DEPLOYMENT_NAME", "gpt-4o")
api_version = os.getenv("API_VERSION", "2024-12-01-preview")

print(f"APIM Endpoint: {apim_endpoint}")
print(f"Deployment: {deployment_name}")

## 기존 코드 vs APIM 경유 코드

기존에 Azure OpenAI를 직접 호출하던 코드:

```python
# 기존 방식 (직접 호출)
client = AzureOpenAI(
    azure_endpoint="https://my-resource.openai.azure.com",
    api_key="foundry-project-api-key",
    api_version="2024-12-01-preview",
)
```

아래와 같이 `azure_endpoint`과 `api_key`만 APIM 값으로 변경하면 됩니다.
나머지 코드는 **완전히 동일**합니다.

In [ ]:
# APIM 경유 호출 — endpoint와 key만 변경
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=apim_endpoint,
    api_key=apim_key,
    api_version=api_version,
)

response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Azure API Management란 무엇인가요? 한 문장으로 설명해주세요."},
    ],
)

print(response.choices[0].message.content)
print(f"\n사용 토큰: {response.usage.total_tokens} (prompt: {response.usage.prompt_tokens}, completion: {response.usage.completion_tokens})")

## 토큰 사용량 확인

위 호출의 토큰 사용량은 **App Insights Telemetry**를 통해 자동으로 기록됩니다.
Azure Portal의 **Cost Dashboard**에서 프로젝트별/모델별 사용량과 예상 비용을 확인할 수 있습니다.

> 💡 대시보드는 Terraform 배포 시 자동 생성됩니다. 별도 설정이 필요 없습니다.

## (확장) 사용자별 추적 — Custom User Header

프로젝트 내 개별 사용자의 토큰 사용량을 추적해야 할 경우,
요청 시 `x-user-id` 헤더를 추가하면 App Insights에 기록됩니다.

```python
response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "user", "content": "안녕하세요"},
    ],
    extra_headers={"x-user-id": "user@company.com"},
)
```

> ⚠️ 이 기능은 초기 배포에는 포함되지 않으며, APIM Policy 확장이 필요합니다.